In [13]:
from support_engineer.boundary import run_agent, AgentRequest, ConfirmationDecision

In [9]:
first = AgentRequest(
    thread_id="1",
    user_id="me",
    message="После деплоя начал падать `billing-api`. В логах много `payment_provider_timeout`. Посмотри runbook, похожие инциденты и предложи план диагностики. Если нужно — заведи incident ticket."
)
response = run_agent(first)

2026-06-11 15:11:56.085 | INFO     | support_engineer.boundary:run_agent:353 - Running support agent: thread_id=1, user_id=me
2026-06-11 15:11:56.090 | INFO     | support_engineer.agent:wrap_model_call:38 - [[Middleware Input]]: Processing query...
2026-06-11 15:12:07.603 | INFO     | support_engineer.agent:wrap_model_call:40 - [[Middleware Output]]: Complete.


In [11]:
response.model_dump()

{'message': '\n\n## Анализ инцидента `billing-api`\n\n### Факты:\n| Источник | Наблюдения |\n|----------|------------|\n| Логи | `payment_provider_timeout` всплеск после деплоя |\n| Инциденты (3 похожих) | - INC-1842: connection pool regression post-deploy<br>- INC-2031: payment-provider latency >2s<br>- INC-2199: false positive от dashboard lag |\n| Memory | После деплоя billing-api timeout spike — чаще всего из-за регресса config (connection pool, retry, timeout) |\n| Runbook | Безопасные проверки: метрики error rate/p95, deploy diff (timeout/retry/pool), активные подключения |\n\n### Гипотезы:\n1. **Связка с деплоем** (вероятность: высокая): config regression — connection pool size reduced / timeout settings изменились  \n2. **Latency payment-provider**: p95 >2s дольше 10 мин (memory: `mem-002`)  \n3. **False positive** (низко): dashboard lag → validated metrics показал проблемы (INC-2199) — маловероятно здесь  \n\n### План диагностики (по runbook, безопасно):\n1. ✅ Сравни deploy di

In [ ]:
second = AgentRequest(
    thread_id="1",
    user_id="me",
    message="Заведи тикет на инцидент и приложи туда этот план в описание"
)
response_second = run_agent(second)

2026-06-11 15:13:46.322 | INFO     | support_engineer.boundary:run_agent:353 - Running support agent: thread_id=1, user_id=me
2026-06-11 15:13:46.326 | INFO     | support_engineer.agent:wrap_model_call:38 - [[Middleware Input]]: Processing query...
2026-06-11 15:13:52.421 | INFO     | support_engineer.agent:wrap_model_call:40 - [[Middleware Output]]: Complete.


AgentResponse(message='Confirmation required before executing create_incident_ticket.', status='confirmation_required', pending_confirmation=PendingConfirmation(confirmation_id='85e2f16361fc3336826f5a9fb86d5a4d', action_name='create_incident_ticket', action_args={'title': 'billing-api payment_provider_timeout spike post-deploy', 'severity': 'SEV-2', 'description': 'Симптомы: billing-api падает с ошибками payment_provider_timeout после деплоя.\n\nDIAGNOSTIC PLAN (по runbook:billing-api):\n1. Безопасные проверки:\n   - Сравнить deploy diff на timeout/retry/connection pool settings\n   - Проверить метрики error rate/p95 latency billing-api\n   - Активные подключения / pending requests / retry rate\n   - Payment-provider latency до/после деплоя\n\n2. Риск-факторы:\n   - Connection pool regression (INC-1842 pattern)\n   - payment-provider p95 >2s >10 мин (INC-2031, mem-002)\n\n3. Unsafe actions требуют явного approval: rollback, restart pods, менять timeout/retry limits\n\nSimilar historica

In [14]:
confirmation = AgentRequest(
    thread_id="1",
    user_id="me",
    decision=ConfirmationDecision(
        confirmation_id="85e2f16361fc3336826f5a9fb86d5a4d", type="approve")
)
response_3 = run_agent(confirmation)

2026-06-11 15:16:07.691 | INFO     | support_engineer.boundary:run_agent:353 - Running support agent: thread_id=1, user_id=me
2026-06-11 15:16:07.695 | WARNING  | support_engineer.tools.local:create_incident_ticket:126 - Created fake incident ticket: INC-FAKE-0001
2026-06-11 15:16:07.710 | INFO     | support_engineer.agent:wrap_model_call:38 - [[Middleware Input]]: Processing query...
2026-06-11 15:16:10.008 | INFO     | support_engineer.agent:wrap_model_call:40 - [[Middleware Output]]: Complete.
